# Simple MS Simulator — Phase 0, G5: offline ICEBERG install on Kaggle

**Generated by `build/mk_simple_sim_gate.py` — do not hand-edit.**

Settings: **GPU T4**, **Internet OFF**. Inputs: the competition data and the dataset
**casmi26-iceberg-offline** (uploaded from `_simlib/gate/kaggle_upload/` by the Colab gate).

It proves the exact thing Phase 4 depends on: ms-pred installs from the wheelhouse with no
network, sees the GPU, and reproduces Colab's predictions for 20 molecules. ~10 min.

In [ ]:
import glob, json, os, subprocess, sys, time
import numpy as np
import pandas as pd

def find_dir(name):
    hits = [h for h in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if os.path.isdir(h)]
    assert hits, f"{name} not found under /kaggle/input -- is casmi26-iceberg-offline attached?"
    return sorted(hits, key=len)[0]

UP = os.path.dirname(find_dir("wheelhouse"))
WH = os.path.join(UP, "wheelhouse")
MAN = json.load(open(os.path.join(UP, "manifest.json")))
print(f"dataset {UP}\nmanifest {MAN}")
print(f"kernel python {sys.version.split()[0]} (wheels built for {MAN['python']})")
print(subprocess.run("nvidia-smi --query-gpu=name,driver_version --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout)
RESULT = {"python_kernel": sys.version.split()[0], "python_wheels": MAN["python"]}

import shutil

# The venv is built from the kernel's interpreter; the wheels are cp312, so the major.minor must match.
VENV = "/kaggle/working/iceberg_venv"
PY = f"{VENV}/bin/python"
py_ver = f"python{sys.version_info.major}.{sys.version_info.minor}"
site_pkg = os.path.join(VENV, "lib", py_ver, "site-packages")

ENV = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME")}
ENV["PYTHONPATH"] = site_pkg
ENV["DGLBACKEND"] = "pytorch"

t0 = time.time()
if not os.path.exists(PY):
    if os.path.exists(VENV):
        shutil.rmtree(VENV, ignore_errors=True)
    # Ubuntu/Kaggle strips ensurepip wheels; --without-pip avoids the ensurepip failure cleanly
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", VENV], check=True)

os.makedirs(site_pkg, exist_ok=True)
wheels = sorted(glob.glob(f"{WH}/*.whl"))
print(f"Installing {len(wheels)} wheels into {site_pkg}...")

# Use host pip to install wheels directly into the venv site-packages
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps",
                    "--target", site_pkg, *wheels],
                   capture_output=True, text=True, env=ENV)
if r.stdout.strip():
    print(r.stdout[-2000:])
if r.stderr.strip():
    print(r.stderr[-2000:])

RESULT["install_ok"] = r.returncode == 0
RESULT["install_s"] = round(time.time() - t0, 1)
assert RESULT["install_ok"], "offline install failed -- G5 FAIL"

r = subprocess.run([PY, "-c", "import torch, dgl, rdkit, ms_pred; "
                    "print(torch.__version__, torch.cuda.is_available(), dgl.__version__, rdkit.__version__)"],
                   capture_output=True, text=True, env=ENV)
print(r.stdout, r.stderr[-3000:])
RESULT["imports_ok"] = r.returncode == 0
last_line = r.stdout.strip().splitlines()[-1].split() if r.stdout.strip() else []
RESULT["cuda"] = (last_line[1] == "True") if (r.returncode == 0 and len(last_line) > 1) else False
print(RESULT)

In [ ]:
import numpy as np
from numba import njit, prange

@njit(cache=True, fastmath=True)
def _clean(mz, it, floor, topk, power, ent_weight):
    n = len(mz)
    if n == 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    mx = 0.0
    for i in range(n):
        if it[i] > mx: mx = it[i]
    if mx <= 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    thr = floor * mx
    c = 0
    for i in range(n):
        if it[i] >= thr: c += 1
    idx = np.empty(c, np.int64)
    j = 0
    for i in range(n):
        if it[i] >= thr:
            idx[j] = i; j += 1
    if c > topk:
        v = np.empty(c, np.float32)
        for i in range(c): v[i] = it[idx[i]]
        o = np.argsort(v)[c - topk:]
        k2 = np.empty(topk, np.int64)
        for i in range(topk): k2[i] = idx[o[i]]
        k2.sort()
        idx = k2
        c = topk
    om = np.empty(c, np.float32)
    oi = np.empty(c, np.float32)
    s = 0.0
    for i in range(c):
        om[i] = mz[idx[i]]
        v = it[idx[i]] ** power
        oi[i] = v
        s += v
    if s > 0:
        for i in range(c): oi[i] /= s
    if ent_weight:
        S = 0.0
        for i in range(c):
            if oi[i] > 0: S -= oi[i] * np.log(oi[i])
        if S < 3.0:
            w = 0.25 + 0.25 * S
            s2 = 0.0
            for i in range(c):
                oi[i] = oi[i] ** w
                s2 += oi[i]
            if s2 > 0:
                for i in range(c): oi[i] /= s2
    return om, oi

@njit(cache=True, fastmath=True)
def entropy_sim(qmz, qp, cmz, cp, tol):
    i = 0; j = 0; n = len(qmz); m = len(cmz)
    SA = 0.0
    for x in range(n):
        if qp[x] > 0: SA -= qp[x] * np.log(qp[x])
    SB = 0.0
    for x in range(m):
        if cp[x] > 0: SB -= cp[x] * np.log(cp[x])
    SAB = 0.0; tot = 0.0
    buf = np.empty(n + m, np.float64); b = 0
    while i < n and j < m:
        d = qmz[i] - cmz[j]
        if d < -tol:
            buf[b] = qp[i]; i += 1; b += 1
        elif d > tol:
            buf[b] = cp[j]; j += 1; b += 1
        else:
            buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1
    while i < n: buf[b] = qp[i]; i += 1; b += 1
    while j < m: buf[b] = cp[j]; j += 1; b += 1
    for x in range(b): tot += buf[x]
    if tot <= 0: return 0.0
    for x in range(b):
        v = buf[x] / tot
        if v > 0: SAB -= v * np.log(v)
    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)

@njit(cache=True, fastmath=True)
def entropy_sim_shift(qmz, qp, cmz, cp, tol, shift):
    """Evaluates max of direct and mass-shifted spectral entropy similarity."""
    a = entropy_sim(qmz, qp, cmz, cp, tol)
    if shift > -0.001 and shift < 0.001:
        return a
    sm = np.empty(len(cmz), np.float32)
    for i in range(len(cmz)):
        sm[i] = cmz[i] + shift
    b = entropy_sim(qmz, qp, sm, cp, tol)
    return a if a > b else b

@njit(cache=True, fastmath=True, parallel=True)
def search(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight):
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]; a = off[c]; b = off[c + 1]
        if b <= a: continue
        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)
        if len(cm) == 0: continue
        out[k] = entropy_sim(qmz, qp, cm, cp, tol)
    return out

@njit(cache=True, fastmath=True, parallel=True)
def search_shift(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight, shift):
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]; a = off[c]; b = off[c + 1]
        if b <= a: continue
        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)
        if len(cm) == 0: continue
        out[k] = entropy_sim_shift(qmz, qp, cm, cp, tol, shift[k])
    return out

In [ ]:
from hashlib import sha1 as _sha1

def sha1(path):
    h = _sha1()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

GEN, INTEN = os.path.join(UP, "ckpt", "gen.ckpt"), os.path.join(UP, "ckpt", "inten.ckpt")
RESULT["sha1_ok"] = sha1(GEN) == MAN["gen_sha1"] and sha1(INTEN) == MAN["inten_sha1"]
assert RESULT["sha1_ok"], "checkpoint SHA1 does not match the Colab gate"

req = pd.read_csv(os.path.join(UP, "smoke_requests.tsv"), sep="	")
out = "/kaggle/working/smoke_resp.npz"
t0 = time.time()
r = subprocess.run([PY, os.path.join(UP, "iceberg_worker.py"), "--gen", GEN, "--inten", INTEN,
                    "--inp", os.path.join(UP, "smoke_requests.tsv"), "--out", out,
                    "--device", "cuda" if RESULT["cuda"] else "cpu"],
                   capture_output=True, text=True, env=ENV)
print(r.stdout[-2000:], r.stderr[-4000:])
assert r.returncode == 0, "worker failed on Kaggle"
RESULT["smoke_wall_s"] = round(time.time() - t0, 1)

z, ref = np.load(out), np.load(os.path.join(UP, "smoke_reference.npz"))
FLOOR, TOPK, POWER, ENTW, TOL = 0.002, 64, 1.0, True, 0.01
def spec(d, i):
    return _clean(d["mz"][d["off"][i]:d["off"][i + 1]].astype(np.float32),
                  d["it"][d["off"][i]:d["off"][i + 1]].astype(np.float32), FLOOR, TOPK, POWER, ENTW)
sims = []
for i in range(len(req)):
    a, b = spec(z, i), spec(ref, i)
    sims.append(float(entropy_sim(a[0], a[1], b[0], b[1], TOL)) if len(a[0]) and len(b[0]) else 0.0)
RESULT["ok_rate"] = float(z["ok"].mean())
RESULT["median_sim_vs_colab"] = float(np.median(sims))
RESULT["min_sim_vs_colab"] = float(np.min(sims))
RESULT["timing"] = json.loads(str(z["timing"]))

passed = (RESULT["install_ok"] and RESULT["imports_ok"] and RESULT["cuda"] and RESULT["sha1_ok"]
          and RESULT["ok_rate"] >= 0.95 and RESULT["median_sim_vs_colab"] >= 0.99)
RESULT["G5"] = "PASS" if passed else "FAIL"
print("=" * 70)
print(json.dumps(RESULT, indent=1))
print("=" * 70)
print(f"G5: {RESULT['G5']}")